# 🌸 EDA Dataset Anggrek — Region of Interest (Unsupervised)

**Tujuan:** Mengeksplorasi dataset gambar anggrek secara unsupervised untuk menemukan
**Region of Interest (ROI)** — area dalam gambar yang mengandung paling banyak perbedaan visual.

## Pipeline
1. [Setup & Load Data](#1)
2. [Statistik Dasar Dataset](#2)
3. [Distribusi Ukuran & Rasio Aspek Gambar](#3)
4. [Distribusi Warna per Kelas](#4)
5. [Variance Map — Area Paling Bervariasi](#5)
6. [Gradient Magnitude Map — Area Tepi & Tekstur](#6)
7. [PCA pada Pixel — Komponen Visual Utama](#7)
8. [K-Means Patch Clustering — Pola Visual Tanpa Label](#8)
9. [t-SNE Embedding — Visualisasi Kesamaan Antar Gambar](#9)
10. [Ringkasan ROI](#10)

---
## 0. Konfigurasi <a id='0'></a>

In [ ]:
# ── Konfigurasi Global ──────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

DATA_DIR      = Path('./data')      # root folder dataset (flow_from_directory style)
IMG_SIZE      = (128, 128)          # resize semua gambar ke resolusi ini
SAMPLE_N      = 200                 # maks gambar yang di-load per kelas (None = semua)
RANDOM_SEED   = 42
N_CLUSTERS    = 6                   # jumlah cluster untuk K-Means patch
PATCH_SIZE    = 16                  # ukuran patch (pixel) untuk clustering

VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
print(f'Dataset root : {DATA_DIR.resolve()}')
print(f'Image size   : {IMG_SIZE}')

---
## 1. Setup & Load Data <a id='1'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from pathlib import Path
import random

sns.set_theme(style='whitegrid', palette='husl')
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Scan folder dataset ──────────────────────────────────────────────────────
def scan_dataset(root: Path, valid_ext=VALID_EXT, sample_n=SAMPLE_N, seed=RANDOM_SEED):
    """Return DataFrame: path, class_name, filename."""
    records = []
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue
        files = [f for f in class_dir.iterdir() if f.suffix.lower() in valid_ext]
        if sample_n and len(files) > sample_n:
            rng = random.Random(seed)
            files = rng.sample(files, sample_n)
        for f in files:
            records.append({'path': f, 'class_name': class_dir.name, 'filename': f.name})
    return pd.DataFrame(records)

df = scan_dataset(DATA_DIR)
print(f'Total gambar ditemukan : {len(df)}')
print(f'Kelas                  : {df["class_name"].nunique()}')
print()
print(df['class_name'].value_counts().rename('jumlah').to_frame())

In [ ]:
# ── Load gambar ke array numpy ───────────────────────────────────────────────
def load_images(df: pd.DataFrame, size=IMG_SIZE):
    """Load & resize semua gambar, return array (N, H, W, C) float32 [0,1]."""
    arrays = []
    failed = []
    for _, row in df.iterrows():
        try:
            img = Image.open(row['path']).convert('RGB').resize(size, Image.LANCZOS)
            arrays.append(np.array(img, dtype=np.float32) / 255.0)
        except Exception as e:
            failed.append((row['path'], str(e)))
    if failed:
        print(f'⚠️  {len(failed)} gambar gagal dimuat.')
    return np.stack(arrays, axis=0)

images = load_images(df)
labels = df['class_name'].values
classes = sorted(df['class_name'].unique())

print(f'Shape array gambar : {images.shape}')   # (N, H, W, 3)
print(f'Dtype              : {images.dtype}')
print(f'Min / Max pixel    : {images.min():.3f} / {images.max():.3f}')

In [ ]:
# ── Sample gambar dari setiap kelas ──────────────────────────────────────────
fig, axes = plt.subplots(len(classes), 5, figsize=(14, 3 * len(classes)))
if len(classes) == 1:
    axes = axes[np.newaxis, :]

for row_idx, cls in enumerate(classes):
    idx = np.where(labels == cls)[0]
    sample_idx = np.random.choice(idx, min(5, len(idx)), replace=False)
    for col_idx, i in enumerate(sample_idx):
        axes[row_idx, col_idx].imshow(images[i])
        axes[row_idx, col_idx].axis('off')
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(cls, fontsize=10, fontweight='bold', rotation=0,
                                               labelpad=80, va='center')
    # kosongkan kolom yang tidak terpakai
    for col_idx in range(len(sample_idx), 5):
        axes[row_idx, col_idx].axis('off')

plt.suptitle('Sample Gambar per Kelas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Statistik Dasar Dataset <a id='2'></a>

In [ ]:
# ── Per-image statistics ──────────────────────────────────────────────────────
brightness  = images.mean(axis=(1, 2, 3))
contrast    = images.std(axis=(1, 2, 3))
r_mean = images[:, :, :, 0].mean(axis=(1, 2))
g_mean = images[:, :, :, 1].mean(axis=(1, 2))
b_mean = images[:, :, :, 2].mean(axis=(1, 2))

stats_df = pd.DataFrame({
    'class_name' : labels,
    'brightness'  : brightness,
    'contrast'    : contrast,
    'r_mean'      : r_mean,
    'g_mean'      : g_mean,
    'b_mean'      : b_mean,
})

print('── Statistik Ringkas per Kelas ──')
display(stats_df.groupby('class_name')[['brightness', 'contrast', 'r_mean', 'g_mean', 'b_mean']]
        .agg(['mean', 'std']).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.violinplot(data=stats_df, x='class_name', y='brightness', ax=axes[0], inner='quartile')
axes[0].set_title('Distribusi Kecerahan (Brightness) per Kelas')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

sns.violinplot(data=stats_df, x='class_name', y='contrast', ax=axes[1], inner='quartile')
axes[1].set_title('Distribusi Kontras (Std Pixel) per Kelas')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

---
## 3. Distribusi Ukuran & Rasio Aspek Gambar Asli <a id='3'></a>

In [ ]:
# ── Baca ukuran gambar ASLI (sebelum resize) ─────────────────────────────────
widths, heights, aspect_ratios = [], [], []
for path in df['path']:
    try:
        with Image.open(path) as im:
            w, h = im.size
            widths.append(w)
            heights.append(h)
            aspect_ratios.append(w / h)
    except:
        widths.append(np.nan)
        heights.append(np.nan)
        aspect_ratios.append(np.nan)

df['width']  = widths
df['height'] = heights
df['aspect'] = aspect_ratios
df['megapixel'] = df['width'] * df['height'] / 1e6

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(df['width'], df['height'], alpha=0.4, s=10,
                c=pd.Categorical(df['class_name']).codes, cmap='tab10')
axes[0].set_xlabel('Lebar (px)'); axes[0].set_ylabel('Tinggi (px)')
axes[0].set_title('Sebaran Dimensi Gambar Asli')

sns.histplot(df['aspect'], kde=True, bins=30, ax=axes[1])
axes[1].set_title('Distribusi Rasio Aspek (W/H)')
axes[1].set_xlabel('Rasio Aspek')

sns.boxplot(data=df, x='class_name', y='megapixel', ax=axes[2])
axes[2].set_title('Megapixel per Kelas')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

---
## 4. Distribusi Warna per Kelas <a id='4'></a>

In [ ]:
# ── Histogram RGB global & per kelas ─────────────────────────────────────────
channel_names = ['Red', 'Green', 'Blue']
channel_colors = ['red', 'green', 'blue']

fig, axes = plt.subplots(len(classes), 3, figsize=(14, 3 * len(classes)), sharey=False)
if len(classes) == 1:
    axes = axes[np.newaxis, :]

bins = np.linspace(0, 1, 64)

for row_idx, cls in enumerate(classes):
    idx = labels == cls
    cls_imgs = images[idx]  # (N_cls, H, W, 3)
    for c, (ch_name, ch_color) in enumerate(zip(channel_names, channel_colors)):
        pixel_vals = cls_imgs[:, :, :, c].ravel()
        axes[row_idx, c].hist(pixel_vals, bins=bins, color=ch_color, alpha=0.7, density=True)
        axes[row_idx, c].set_xlim(0, 1)
        if row_idx == 0:
            axes[row_idx, c].set_title(f'Kanal {ch_name}', fontweight='bold')
        if c == 0:
            axes[row_idx, c].set_ylabel(cls, fontsize=9, fontweight='bold')

plt.suptitle('Histogram Warna (RGB) per Kelas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Rata-rata dominansi warna antar kelas (HSV) ───────────────────────────────
from PIL import ImageStat

hsv_stats = []
for i, (img_arr, cls) in enumerate(zip(images, labels)):
    pil_img = Image.fromarray((img_arr * 255).astype(np.uint8)).convert('HSV')
    stat = ImageStat.Stat(pil_img)
    hsv_stats.append({'class_name': cls, 'H_mean': stat.mean[0],
                      'S_mean': stat.mean[1], 'V_mean': stat.mean[2]})

hsv_df = pd.DataFrame(hsv_stats)
palette = sns.color_palette('husl', len(classes))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['H_mean', 'S_mean', 'V_mean']):
    sns.boxplot(data=hsv_df, x='class_name', y=col, ax=ax, palette=palette)
    ax.set_title(col)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Distribusi HSV per Kelas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Variance Map — Area Paling Bervariasi (ROI Kandidat Utama) <a id='5'></a>

> **Ide:** Hitung **varians setiap piksel** di seluruh dataset (atau per kelas). Piksel yang
> variansinya tinggi = area yang paling berbeda antar gambar → **kandidat ROI**.

In [ ]:
# ── Variance map global ───────────────────────────────────────────────────────
var_map_global = images.var(axis=0)               # (H, W, 3)
var_map_gray   = var_map_global.mean(axis=-1)     # (H, W) — gabungkan channel

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(var_map_gray, cmap='hot', interpolation='nearest')
axes[0].set_title('Variance Map Global\n(semua kelas)', fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Overlay threshold 75th-percentile sebagai kontur ROI
threshold = np.percentile(var_map_gray, 75)
roi_mask  = var_map_gray >= threshold
mean_img  = images.mean(axis=0)                   # rata-rata visual

axes[1].imshow(mean_img)
axes[1].contour(roi_mask, levels=[0.5], colors='yellow', linewidths=1.5)
axes[1].set_title(f'Mean Image + ROI Kontur\n(varians ≥ p75 = {threshold:.4f})', fontweight='bold')
axes[1].axis('off')

plt.suptitle('Variance Map — ROI Kandidat', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Area ROI  : {roi_mask.sum()} piksel ({roi_mask.mean()*100:.1f}% gambar)')

In [ ]:
# ── Variance map per kelas ────────────────────────────────────────────────────
n_cls = len(classes)
ncols = min(n_cls, 4)
nrows = (n_cls + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.array(axes).flatten()

vmax = 0
cls_var_maps = {}
for cls in classes:
    cls_imgs = images[labels == cls]
    vmap = cls_imgs.var(axis=0).mean(axis=-1)
    cls_var_maps[cls] = vmap
    vmax = max(vmax, vmap.max())

for i, cls in enumerate(classes):
    im = axes[i].imshow(cls_var_maps[cls], cmap='hot', vmin=0, vmax=vmax,
                        interpolation='nearest')
    axes[i].set_title(cls, fontsize=10, fontweight='bold')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Variance Map per Kelas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Kuantifikasi: region paling variatif (grid 4×4) ───────────────────────────
H, W = IMG_SIZE
GRID  = 4   # bagi gambar ke grid 4×4 = 16 sel
cell_h = H // GRID
cell_w = W // GRID

grid_var = np.zeros((GRID, GRID))
for r in range(GRID):
    for c in range(GRID):
        patch = var_map_gray[r*cell_h:(r+1)*cell_h, c*cell_w:(c+1)*cell_w]
        grid_var[r, c] = patch.mean()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(grid_var, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=[f'Col {i+1}' for i in range(GRID)],
            yticklabels=[f'Row {i+1}' for i in range(GRID)],
            ax=ax, linewidths=0.5)
ax.set_title(f'Rata-rata Varians per Sel Grid {GRID}×{GRID}\n'
             f'(Nilai lebih tinggi = area lebih bervariasi = ROI kandidat)',
             fontweight='bold')
plt.tight_layout()
plt.show()

top_r, top_c = np.unravel_index(grid_var.argmax(), grid_var.shape)
print(f'Sel grid paling bervariasi: Baris {top_r+1}, Kolom {top_c+1}')
print(f'  → area piksel: y=[{top_r*cell_h}:{(top_r+1)*cell_h}], x=[{top_c*cell_w}:{(top_c+1)*cell_w}]')

---
## 6. Gradient Magnitude Map — Area Tepi & Tekstur <a id='6'></a>

> **Ide:** Area dengan gradien tinggi = tepi tajam, tekstur kompleks → kemungkinan ROI yang informatif.

In [ ]:
from skimage.filters import sobel
from skimage.color import rgb2gray

# ── Hitung rata-rata gradient magnitude seluruh gambar ────────────────────────
grad_maps = []
for img in images:
    gray = rgb2gray(img)
    grad_maps.append(sobel(gray))

grad_maps = np.stack(grad_maps, axis=0)   # (N, H, W)
mean_grad = grad_maps.mean(axis=0)        # (H, W)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(mean_grad, cmap='inferno')
axes[0].set_title('Rata-rata Gradient Magnitude (Global)', fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

grad_threshold = np.percentile(mean_grad, 75)
grad_roi = mean_grad >= grad_threshold
axes[1].imshow(mean_img)
axes[1].contour(grad_roi, levels=[0.5], colors='cyan', linewidths=1.5)
axes[1].set_title(f'Mean Image + Gradient ROI\n(gradient ≥ p75 = {grad_threshold:.4f})', fontweight='bold')
axes[1].axis('off')

plt.suptitle('Gradient Magnitude Map — ROI Tekstur', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gabungkan Variance ROI + Gradient ROI ────────────────────────────────────
combined_roi = roi_mask & grad_roi

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(mean_img)
axes[0].contour(roi_mask, levels=[0.5], colors='yellow', linewidths=1.5)
axes[0].set_title('ROI: Variance Map\n(kuning)', fontweight='bold'); axes[0].axis('off')

axes[1].imshow(mean_img)
axes[1].contour(grad_roi, levels=[0.5], colors='cyan', linewidths=1.5)
axes[1].set_title('ROI: Gradient Map\n(cyan)', fontweight='bold'); axes[1].axis('off')

axes[2].imshow(mean_img)
axes[2].contour(combined_roi, levels=[0.5], colors='lime', linewidths=2)
axes[2].set_title('ROI: Gabungan\n(hijau = variance AND gradient)', fontweight='bold'); axes[2].axis('off')

plt.suptitle('Perbandingan Metode ROI Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Overlap ROI (variance ∩ gradient): {combined_roi.sum()} piksel '
      f'({combined_roi.mean()*100:.1f}% area gambar)')

---
## 7. PCA pada Pixel — Komponen Visual Utama <a id='7'></a>

> **Ide:** Flatten setiap gambar menjadi vektor, lakukan PCA. Komponen pertama
> menangkap variasi terbesar di seluruh dataset. Visualisasi eigenvector = "eigenface" anggrek.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ── Flatten gambar ────────────────────────────────────────────────────────────
N = len(images)
X_flat = images.reshape(N, -1)   # (N, H*W*3)
print(f'Shape matriks fitur : {X_flat.shape}')

# ── PCA ───────────────────────────────────────────────────────────────────────
n_components = min(50, N - 1)
pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_flat)

# Explained variance
cumvar = np.cumsum(pca.explained_variance_ratio_)
n90 = np.searchsorted(cumvar, 0.90) + 1

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, n_components + 1), pca.explained_variance_ratio_,
       alpha=0.7, label='Per-komponen')
ax.plot(range(1, n_components + 1), cumvar, 'r-o', markersize=3,
        label='Kumulatif')
ax.axhline(0.90, color='gray', linestyle='--', label='90% threshold')
ax.axvline(n90, color='gray', linestyle=':')
ax.set_xlabel('Komponen PCA'); ax.set_ylabel('Rasio Variansi')
ax.set_title('Explained Variance Ratio — PCA', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Komponen untuk 90% variansi: {n90}')

In [ ]:
# ── Visualisasi Eigenvector (PC) sebagai Gambar ───────────────────────────────
n_show = min(8, n_components)
fig, axes = plt.subplots(2, n_show, figsize=(2.5 * n_show, 5))

for i in range(n_show):
    ev = pca.components_[i].reshape(*IMG_SIZE, 3)
    # Normalisasi ke [0,1] untuk visualisasi
    ev_norm = (ev - ev.min()) / (ev.max() - ev.min() + 1e-8)
    axes[0, i].imshow(ev_norm)
    axes[0, i].set_title(f'PC{i+1}\n({pca.explained_variance_ratio_[i]*100:.1f}%)',
                         fontsize=9)
    axes[0, i].axis('off')

    # Magnitude variasi per piksel untuk PC ini
    mag = np.abs(ev).mean(axis=-1)
    im = axes[1, i].imshow(mag, cmap='hot')
    axes[1, i].set_title(f'|PC{i+1}|', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('Eigenvector (Principal Components) — Atas: RGB | Bawah: Magnitude',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter Plot PC1 vs PC2 per kelas ────────────────────────────────────────
palette = sns.color_palette('tab10', len(classes))
color_map = {cls: palette[i] for i, cls in enumerate(classes)}

fig, ax = plt.subplots(figsize=(9, 7))
for cls in classes:
    idx = labels == cls
    ax.scatter(X_pca[idx, 0], X_pca[idx, 1],
               label=cls, alpha=0.6, s=20, color=color_map[cls])
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA — PC1 vs PC2 (per kelas)', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

---
## 8. K-Means Patch Clustering — Pola Visual Tanpa Label <a id='8'></a>

> **Ide:** Ekstrak patch kecil dari semua gambar, cluster menggunakan K-Means.
> Visualisasi centroid = **pola visual yang paling sering muncul** → petunjuk ROI informatif.

In [ ]:
from sklearn.cluster import MiniBatchKMeans

# ── Ekstrak patch secara acak dari setiap gambar ──────────────────────────────
PATCHES_PER_IMG = 30
PS = PATCH_SIZE
H, W = IMG_SIZE

all_patches = []
patch_origins = []   # (image_idx, y, x)

rng = np.random.default_rng(RANDOM_SEED)
for img_idx, img in enumerate(images):
    ys = rng.integers(0, H - PS, PATCHES_PER_IMG)
    xs = rng.integers(0, W - PS, PATCHES_PER_IMG)
    for y, x in zip(ys, xs):
        patch = img[y:y+PS, x:x+PS, :]   # (PS, PS, 3)
        all_patches.append(patch.ravel())
        patch_origins.append((img_idx, y, x))

all_patches = np.stack(all_patches, axis=0)
print(f'Total patch diekstrak : {len(all_patches)} ({all_patches.shape})')

# ── K-Means clustering ────────────────────────────────────────────────────────
kmeans = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED,
                         n_init=5, max_iter=200)
cluster_labels = kmeans.fit_predict(all_patches)
print(f'Cluster terbentuk: {N_CLUSTERS}')
unique, counts = np.unique(cluster_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Cluster {u}: {c} patch ({c/len(cluster_labels)*100:.1f}%)')

In [ ]:
# ── Visualisasi centroid cluster ──────────────────────────────────────────────
fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(3 * N_CLUSTERS, 3))
for k in range(N_CLUSTERS):
    centroid = kmeans.cluster_centers_[k].reshape(PS, PS, 3)
    centroid = np.clip(centroid, 0, 1)
    axes[k].imshow(centroid, interpolation='nearest')
    n_patches_k = (cluster_labels == k).sum()
    axes[k].set_title(f'Cluster {k}\nn={n_patches_k}', fontsize=9)
    axes[k].axis('off')

plt.suptitle(f'Centroid K-Means Patch Clustering (K={N_CLUSTERS})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap frekuensi cluster per posisi (spatial distribution) ───────────────
# Bagi gambar ke grid cell, hitung frekuensi tiap cluster di tiap cell
GRID_K = 8
cell_h = H // GRID_K
cell_w = W // GRID_K

cluster_spatial = np.zeros((N_CLUSTERS, GRID_K, GRID_K))
for (img_idx, y, x), k in zip(patch_origins, cluster_labels):
    r = min(y // cell_h, GRID_K - 1)
    c = min(x // cell_w, GRID_K - 1)
    cluster_spatial[k, r, c] += 1

ncols = min(N_CLUSTERS, 3)
nrows = (N_CLUSTERS + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
axes = np.array(axes).flatten()

for k in range(N_CLUSTERS):
    sns.heatmap(cluster_spatial[k], ax=axes[k], cmap='YlOrRd',
                cbar=True, linewidths=0.3)
    axes[k].set_title(f'Cluster {k} — Distribusi Spasial', fontsize=10, fontweight='bold')

for j in range(N_CLUSTERS, len(axes)):
    axes[j].axis('off')

plt.suptitle('Distribusi Spasial Tiap Cluster Patch\n(Warna panas = banyak patch cluster ini di area tersebut)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. t-SNE Embedding — Visualisasi Kesamaan Antar Gambar <a id='9'></a>

> **Ide:** Proyeksikan setiap gambar ke 2D menggunakan t-SNE. Gambar yang secara visual
> mirip akan berdekatan → tampilkan thumbnail di posisi t-SNE untuk insight intuitif.

In [ ]:
from sklearn.manifold import TSNE

# Gunakan skor PCA (lebih efisien daripada raw pixel)
n_pca_for_tsne = min(30, X_pca.shape[1])
X_tsne_input = X_pca[:, :n_pca_for_tsne]

print(f't-SNE input shape: {X_tsne_input.shape}')
print('Menjalankan t-SNE... (mungkin 1-2 menit untuk dataset besar)')

tsne = TSNE(n_components=2, perplexity=min(30, N // 5),
            random_state=RANDOM_SEED, n_iter=1000, verbose=0)
X_2d = tsne.fit_transform(X_tsne_input)
print(f't-SNE selesai. Shape: {X_2d.shape}')

In [ ]:
# ── Scatter t-SNE (titik berwarna per kelas) ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for cls in classes:
    idx = labels == cls
    ax.scatter(X_2d[idx, 0], X_2d[idx, 1],
               label=cls, alpha=0.7, s=25, color=color_map[cls])
ax.set_title('t-SNE Embedding Gambar Anggrek', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

In [ ]:
# ── Thumbnail t-SNE (tampilkan gambar mini di posisi embedding) ───────────────
# Batasi jumlah thumbnail agar tidak terlalu padat
MAX_THUMB = min(150, N)
thumb_idx = np.random.choice(N, MAX_THUMB, replace=False)
THUMB_SIZE = 20   # ukuran thumbnail dalam unit axes

fig, ax = plt.subplots(figsize=(14, 11))
ax.set_facecolor('#1a1a2e')

x_range = X_2d[:, 0].max() - X_2d[:, 0].min()
y_range = X_2d[:, 1].max() - X_2d[:, 1].min()
thumb_w = x_range * 0.04
thumb_h = y_range * 0.04

for i in thumb_idx:
    x, y = X_2d[i, 0], X_2d[i, 1]
    img_thumb = Image.fromarray((images[i] * 255).astype(np.uint8)).resize((32, 32))
    from matplotlib.offsetbox import OffsetImage, AnnotationBbox
    imagebox = OffsetImage(np.array(img_thumb), zoom=0.5)
    ab = AnnotationBbox(imagebox, (x, y), frameon=True,
                        bboxprops=dict(edgecolor=color_map[labels[i]], linewidth=1.5, alpha=0.8))
    ax.add_artist(ab)

ax.set_xlim(X_2d[:, 0].min() - thumb_w * 2, X_2d[:, 0].max() + thumb_w * 2)
ax.set_ylim(X_2d[:, 1].min() - thumb_h * 2, X_2d[:, 1].max() + thumb_h * 2)
ax.set_title(f't-SNE Thumbnail Grid ({MAX_THUMB} gambar)\n'
             'Border warna = kelas', fontsize=12, fontweight='bold', color='white')
ax.tick_params(colors='white'); ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')

patches_legend = [mpatches.Patch(color=color_map[cls], label=cls) for cls in classes]
ax.legend(handles=patches_legend, bbox_to_anchor=(1.02, 1), loc='upper left',
          fontsize=9, facecolor='#2d2d44', labelcolor='white')
plt.tight_layout()
plt.show()

---
## 10. Ringkasan ROI <a id='10'></a>

In [ ]:
# ── Skor ROI Gabungan: variance + gradient + PC1 magnitude ───────────────────
# Normalkan masing-masing ke [0,1] lalu rata-ratakan
def normalize(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

pc1_magnitude = np.abs(pca.components_[0]).reshape(*IMG_SIZE, 3).mean(axis=-1)

roi_score = (
    normalize(var_map_gray) * 0.40 +
    normalize(mean_grad)    * 0.35 +
    normalize(pc1_magnitude)* 0.25
)

roi_threshold_final = np.percentile(roi_score, 80)
final_roi_mask = roi_score >= roi_threshold_final

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

im0 = axes[0].imshow(roi_score, cmap='RdYlGn')
axes[0].set_title('ROI Score Map\n(40% var + 35% grad + 25% PC1)', fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

axes[1].imshow(mean_img)
axes[1].contourf(final_roi_mask, levels=[0.5, 1], colors=['yellow'], alpha=0.4)
axes[1].contour(final_roi_mask, levels=[0.5], colors='red', linewidths=2)
axes[1].set_title('Mean Image + Final ROI\n(top 20% score, kontur merah)', fontweight='bold')
axes[1].axis('off')

# Terapkan ROI mask ke gambar asli sampel
sample_img = images[0].copy()
dimmed = sample_img * 0.3
roi_3ch = np.stack([final_roi_mask]*3, axis=-1)
highlighted = np.where(roi_3ch, sample_img, dimmed)
axes[2].imshow(highlighted)
axes[2].set_title('Contoh Highlight ROI pada Gambar', fontweight='bold')
axes[2].axis('off')

plt.suptitle('🎯 Final Region of Interest (ROI)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── ROI per kelas: highlight di atas mean gambar kelas ────────────────────────
nrows = (len(classes) + 2) // 3
ncols = min(len(classes), 3)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = np.array(axes).flatten()

for i, cls in enumerate(classes):
    idx = labels == cls
    cls_imgs = images[idx]
    cls_mean = cls_imgs.mean(axis=0)
    cls_var  = cls_imgs.var(axis=0).mean(axis=-1)
    cls_grad = grad_maps[idx].mean(axis=0)

    cls_score = (
        normalize(cls_var)  * 0.50 +
        normalize(cls_grad) * 0.50
    )
    cls_roi = cls_score >= np.percentile(cls_score, 80)

    dimmed = cls_mean * 0.3
    roi_3ch = np.stack([cls_roi]*3, axis=-1)
    highlighted = np.where(roi_3ch, cls_mean, dimmed)

    axes[i].imshow(highlighted)
    axes[i].contour(cls_roi, levels=[0.5], colors='lime', linewidths=1.5)
    axes[i].set_title(f'{cls}\n(ROI = {cls_roi.mean()*100:.1f}% area)', fontsize=10, fontweight='bold')
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('ROI per Kelas (area paling informatif di-highlight)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Ringkasan Statistik ROI ───────────────────────────────────────────────────
print('=' * 55)
print('  RINGKASAN EDA — REGION OF INTEREST ANGGREK')
print('=' * 55)
print(f'\nTotal gambar dianalisis : {N}')
print(f'Kelas                   : {len(classes)}')
print(f'Resolusi analisis       : {IMG_SIZE[0]}×{IMG_SIZE[1]} px')
print()
print('── Variance Map ──')
print(f'  Piksel varians tertinggi ada di grid sel ({top_r+1},{top_c+1})')
print(f'  → Y: {top_r*cell_h}–{(top_r+1)*cell_h}px, X: {top_c*cell_w}–{(top_c+1)*cell_w}px')
print()
print('── PCA ──')
print(f'  PC1 menjelaskan {pca.explained_variance_ratio_[0]*100:.1f}% variansi')
print(f'  {n90} komponen cukup untuk 90% variansi')
print()
print('── K-Means Patch Clustering ──')
print(f'  {N_CLUSTERS} cluster visual ditemukan')
top_cluster = np.bincount(cluster_labels).argmax()
print(f'  Cluster dominan: #{top_cluster} ({np.bincount(cluster_labels)[top_cluster]/len(cluster_labels)*100:.1f}% patch)')
print()
print('── Final ROI Score ──')
roi_bbox = np.where(final_roi_mask)
if len(roi_bbox[0]) > 0:
    y_min, y_max = roi_bbox[0].min(), roi_bbox[0].max()
    x_min, x_max = roi_bbox[1].min(), roi_bbox[1].max()
    print(f'  Bounding box ROI: y=[{y_min},{y_max}], x=[{x_min},{x_max}]')
    print(f'  Ukuran ROI      : {y_max-y_min}×{x_max-x_min} px')
    print(f'  Coverage ROI    : {final_roi_mask.mean()*100:.1f}% area gambar')
print()
print('✅ EDA selesai. ROI siap digunakan untuk:')
print('   - Crop / attention preprocessing')
print('   - Focal loss weighting')
print('   - Guided augmentation')